In [4]:
from pathlib import Path
import polars as pl

In [10]:
out_dir = Path("../02_experiment/out")
models = ["openai_gpt-5.6-luna", "deepseek_deepseek-v4-flash-0731", "qwen_qwen3.6-35b-a3b"]

results = {
    (model, codebook): pl.read_csv(out_dir / f"model_{model}_codebook_{codebook}_codebook_labels.csv")
        .with_columns(pl.lit(model).alias("model"), pl.lit(codebook).alias("codebook"))
    for model in models for codebook in models
}

all_labels = pl.concat(results.values(), how="diagonal_relaxed")

gold = pl.read_csv("../02_experiment/data/populism_codebookapply.csv")

In [11]:
all_labels_wide = (
    all_labels
    .with_columns(pl.concat_str("model", "codebook", separator="__").alias("run"))
    .pivot(on="run", index=["doc_id", "text"], values=["label", "confidence", "decision_basis"])
)

all_labels_wide = all_labels_wide.join(gold, on="doc_id")
all_labels_wide.write_csv("./out/all_labels_wide.csv")